# ABI 7500 Fast Dx: connection, experimental thermal runs and offline data

```{device-card} thermo-fisher-abi7500-fast-dx
```

**Incomplete integration.** Connection, identity, and temperature/status queries were checked directly from Ubuntu on one G2.10 instrument. A 35 °C/60-second hold completed normally, and a separate start/cancel succeeded. Two 35 °C/30 °C cycles also completed normally with 30 hold-associated images. Broader programs, startup cancellation and fault recovery remain hardware unverified. Illuminated snapshots at filters 1 and 5, lamp power and shutdown succeeded from Ubuntu. Offline processing reconstructs all 960 saved well integers and their corrected values from one recorded vendor acquisition. Optical fault recovery and current calibration validity remain unverified. Read the [hardware report](hardware-validation.md) and [run-control limits](thermal-runs.md) before using hardware cells.

| Property | Value |
|---|---|
| Transport | USB through the Hamamatsu camera bridge |
| Plate | 96 wells |
| Hardware-tested operations | Connect, identify, query status/history, load, start, observe, cancel, complete a 35 °C/60-second hold, raw snapshots, lamp enable/power/disable and filter 0–5 readback, two synchronized cycles with 30 images, disconnect |
| Experimental run scope | Standard mode, 20 µL, block 2, finite holds; optional scheduled collection |
| Validation | Direct Ubuntu observation, one start/cancel and one completed hold and two completed optical cycles; mocked fault tests |


The status-scheduled collection API completed a two-cycle run normally with 30 complete images.
The approved session used a 900-second observation budget and restored lamp off and filter 0. See
[synchronized acquisition](synchronized-acquisition.md) for timing, result and recovery limits.

For a complete finite PCR-temperature method with progress, cooling, release and original-image export, use [Run a PCR plate](pcr-run.ipynb).

## Connection and physical setup

The Hamamatsu USB connection carries camera messages and a tunnel for thermal-controller packets. Close the Windows 7500 software and ensure no experiment is running before moving the instrument's USB cable to the PLR computer. Keep the instrument powered and use the existing USB Type-B connection. The EZ-Tap RS-232 device is not part of this USB connection.

Install PLR's USB dependencies and give your account access to the instrument's USB device. Do not run the hardware cells without the required local authorization. The packet examples later in this notebook run offline.


## Open the USB connection

This hardware cell configures the communication bridge and checks identification. The driver warns that the integration is incomplete and identifies the limited hardware-tested scope. It does not upload a thermal program, start heating, home filters, or set lamp power.

In [ ]:
from pathlib import Path

from pylabrobot.thermo_fisher import ABI7500FastDx

instrument = ABI7500FastDx(run_journal=Path.home() / ".local/state/pylabrobot/abi7500/runs.sqlite3")
await instrument.setup()

## Query identity

The camera identification and thermal firmware version refer to different components. Preserve their complete replies when documenting a hardware test.

In [ ]:
identity = await instrument.get_identity()
identity.camera_description, identity.thermal_version

## Query temperature and sequence status

Temperatures are in Celsius and elapsed time is in seconds. Native indices and hold time may retain previous-run values while inactive; elapsed time can continue advancing. An inactive sequence does not prove successful completion or that the cover is cool.

In [ ]:
status = await instrument.get_status()
{
  "block": status.block_temperature,
  "sample": status.sample_temperature,
  "cover": status.cover_temperature,
  "ambient": status.ambient_temperature,
  "phase": status.phase,
  "elapsed": status.elapsed_time,
}

## Read thermal history

This query was checked directly from Ubuntu. History can describe an earlier run; its outcome alone does not establish the state of a PLR run. `no_result` (code 253) can occur during an active run.

In [ ]:
history = await instrument.get_history()
print(history.outcome, history.method_name, history.duration)

## Query the internal plate mechanism

This read-only query reports `loaded`, `moving`, `unloaded`, or `unknown`. An unloaded mechanism
does not prove that the external drawer is open, that a plate is present, or that the cover is cool.

In [ ]:
plate = await instrument.get_plate_status()
print(plate.state, plate.state_code, plate.raw_payload.hex())

## Read stored vendor exposure times

Within the approved connection session, query the instrument’s three stored exposures per physical filter. Durations are in seconds, in the original slot order. Physical filter 5 corresponds to saved channel 1 (SYBR/FAM) on the reference instrument. This readback does not change settings, select exposures for a PLR collection plan, or establish calibration validity. Two {ref}`direct Ubuntu readbacks on G2.10 <abi7500-exptable-validation>` matched the complete recorded vendor table. Captured-transport tests cover both vendor and Ubuntu transactions.

In [ ]:
stored_exposures = await instrument.get_stored_exposure_times()
for physical_filter in range(1, 6):
    print(physical_filter, stored_exposures.for_filter(physical_filter))

## Optional experimental thermal run: build the program offline

The following optional hardware cells move the internal loading mechanism and heat the instrument. Execute them only for an explicitly approved engineering session with a sealed plate ready to load. The block target is 35 °C for 60 seconds; the instrument's cover behavior can still heat the cover to about 105 °C. No fluorescence is collected.

This offline cell retains the captured Standard/20 µL/block-2/tube-0 header and native date metadata, changing only the thermal steps. Run submission will replace the method/user labels with the run ID and `PLR`. Other modes and volumes are not supported by this experimental API.

In [ ]:
from pylabrobot.thermo_fisher.abi7500_fast_dx import ThermalCycle, ThermalProgram, ThermalStep

reference_metadata = bytes.fromhex(
  "4d736d616c6c00000000000000000000004d617961000000630a02020a0b0c0000620a02020a030100001400020000"
)
program = ThermalProgram(
  metadata=reference_metadata,
  cycles=(ThermalCycle(repetitions=1, steps=(ThermalStep(temperature=35.0, hold_time=60),)),),
)

## Submit one thermal-only run

Use a unique ID for each intended experiment and retain the same ID/program/journal for retries. Repeating this cell returns the existing checkpoint without resending Start. Do not change IDs or journal paths to bypass an uncertain result. `starting` means an acknowledgement or submission checkpoint, not confirmed heating.

In [ ]:
run_id = "plr-thermal-001"
run = await instrument.start_run(run_id, program)
print(run.state)

## Observe run progress

Repeat this cell to obtain fresh status/history evidence. Initial status can still belong to the preceding run. `uncertain` requires inspection; it is not permission to retry Start. A thermal `completed` result does not indicate fluorescence acquisition or that the cover is cool.

Startup can take several minutes while the cover warms: the first PLR cancellation test took about 273 seconds to report fresh active progress. Keep the same run ID and journal. A short observation timeout does not cancel the pending run or justify sending Start again.

To finish naturally, continue observing until `completed`. The checked 35 °C/60-second run reported a 73-second total duration including ramp. It remained at 35 °C after completion; normal completion does not send a 25 °C idle request.

In [ ]:
run = await instrument.get_run(run_id)
print(run.state, run.last_status, run.history)

## Explicitly cancel the identified run

For a cancellation test, call this after a fresh `running` result. Cancellation confirms inactivity before requesting the 25 °C idle target. It does not eject the plate or cool the cover, and a successful target command does not mean 25 °C has been reached. Unknown run ownership is refused. The current API cannot cancel startup before a fresh owned run record appears.

In [ ]:
run = await instrument.cancel_run(run_id)
print(run.state, run.last_status)

## Reconcile after a communication failure

Use this recovery cell only if needed, with the same journal and run ID. It reconnects and queries evidence without starting or stopping a sequence. An unresolved outcome stays unresolved; do not delete the checkpoint. `stop()` itself does not cancel heating.

In [ ]:
await instrument.stop()
await instrument.setup()
run = await instrument.get_run(run_id)
print(run.state)

## Optional raw camera snapshot

This experimental operation needs its own hardware approval. Snapshots at filter indices 0, 1 and 5 succeeded; illuminated wells were observed at filters 1 and 5. Filter 5 saturated at 0.090 seconds. Optical fault recovery remains unverified. Use it only with an inactive thermal controller and resolved journaled runs. It takes one 0.256-second exposure using the current lamp/filter settings, without heating or moving the plate/filter. The image may be dark. Reusing the acquisition ID returns the existing record, including uncertainty after failure; a new ID requests a separate exposure. See [snapshot limits](raw-snapshots.md).

In [ ]:
acquisition_id = "plr-snapshot-001"
acquisition = await instrument.capture_frame(acquisition_id, exposure=0.256)
print(acquisition.state, acquisition.optics_before, acquisition.error)

## Inspect the stored frame

Reading the journal performs no USB I/O. Check the acquisition state separately from frame completeness: a full frame can survive a failed final status query. These are original image pixels, not calibrated well measurements.

In [ ]:
acquisition = await instrument.get_acquisition(acquisition_id)
frame = acquisition.frame
print(acquisition.state, None if frame is None else (frame.width, frame.height))
if frame is not None:
  print(len(frame.wire_data), len(frame.image_bytes))

## Optional lamp/filter control

These setters require approval for an optical hardware session. Filter 0/1 and lamp enable/disable readback were checked directly from Ubuntu;
the lamp-enabled image remained near the earlier dark-background level. The instrument must be idle and the journal's thermal runs resolved.
Native filter position 1 and lamp power code 0 follow the captured vendor acquisition; they do not
specify a dye or calibrated brightness. Keep the lamp enabled only for the approved session.
`stop()` does not switch it off. After a failed exchange, reconnect and inspect state before any
further action; no automatic restoration is attempted after an ambiguous failure.

This block does not acquire images or start a thermal run. See
[optical control and measurements](optics-and-measurements.md).

In [ ]:
before_optics = await instrument.get_optical_state()
print(before_optics)
optics = await instrument.set_filter_position(1)
optics = await instrument.set_lamp(True, power=0)
print(optics)

## End the approved lamp session

Execute this explicitly after the authorized observations. Disabling the lamp does not cool the
heated cover. Restoring the previous filter position remains within the approved session.

In [ ]:
await instrument.set_lamp(False)
await instrument.set_filter_position(before_optics.filter_position)

## Optional synchronized collection: prepare offline

This separate engineering session needs per-run approval. A two-cycle run completed normally with
30 images using a 900-second observation budget. This example requests the same mild cycles,
with 15 images during each 30 °C hold. These are physical filters, not dye labels. Every 60-second
collection hold must accommodate the full 45-second read window. The plate must remain sealed and
ready for loading. The cover can remain near 105 °C. Broader programs and fault recovery remain unverified.

This plan parks at absolute filter position 0 after each burst and verifies stationary readback with lamp power 1 retained. The move must fit inside the hold. This is not the vendor’s collection-end home command; a failed park retains frames but makes the collection uncertain. Older plans default to no parking.

`home_after_collections` selects complete collection ordinals, not thermal step indices. For this two-collection example, `(1, 2)` would request a native home after each burst instead of absolute parking. Keep the empty tuple unless homing is part of the approved session. PLR saves the intent before movement and never retries an uncertain home.

In [ ]:
from pylabrobot.thermo_fisher.abi7500_fast_dx import CollectionFilter, CollectionPlan

optical_program = ThermalProgram(reference_metadata, (
  ThermalCycle(2, (ThermalStep(35, 15), ThermalStep(30, 60))),
))
optical_plan = CollectionPlan(
  steps=((0, 1),),
  filters=tuple(CollectionFilter(position, exposures) for position, exposures in (
    (1, (0.540, 0.180, 0.060)), (2, (0.540, 0.180, 0.060)),
    (3, (0.900, 0.300, 0.100)), (4, (0.540, 0.255, 0.085)),
    (5, (1.048, 0.090, 0.030)),
  )),
  read_window=45,
  park_between_collections=True,
  home_after_collections=(),  # Explicit one-based ordinals opt into native homing.
)
optical_plan.points(optical_program)

## Execute the approved optical session and clean up

Set the guard only after approval for this run. Keep Start and the collector in the same cell:
starting the thermal method alone does not schedule images. A unique ID means a new experiment;
never change it merely to retry an uncertain result. Keep the same journal.

Cleanup reconnects once if needed and cancels only a positively owned active run. After confirmed
inactivity it disables the lamp, parks filter 0, and calls `release_plate()` for the same finished
run. Release checks fresh identity/history and commits its intent before moving the internal
mechanism. Keep hands clear; unloaded does not establish that the external drawer is open or cool.
The connection closes even when cleanup raises. Inspect unresolved evidence instead of replaying
the session. A `not_started` result does not authorize release after failed loading.

If a prior release is uncertain, cleanup calls its observation-only reconciliation directly;
other actuation remains blocked. These hardware cells are never executed by documentation tests.

In [ ]:
import asyncio
from pylabrobot.thermo_fisher.abi7500_fast_dx import ABI7500RunError

execute_approved_optical_session = False
if not execute_approved_optical_session:
  raise RuntimeError("This optional session requires explicit per-run approval")

optical_run_id = "plr-optical-001"
start_attempted = False
try:
  await instrument.set_filter_position(0)
  await instrument.set_lamp(True, power=1)
  await asyncio.sleep(5)
  start_attempted = True
  await instrument.start_run(optical_run_id, optical_program, collection_plan=optical_plan)
  collections = await instrument.collect_fluorescence(optical_run_id, timeout=900)
  print([(c.record.point, c.record.state, len(c.acquisitions)) for c in collections])
finally:
  try:
    if not instrument.is_connected:
      await instrument.stop()
      await instrument.setup()
    optical_run = None
    if start_attempted:
      try:
        optical_run = await instrument.get_run(optical_run_id)
      except ABI7500RunError:
        # A failure before reservation has no run to release. Optical setters still
        # independently require matching identity, a clear journal and inactivity.
        optical_run = None
      if optical_run is not None and not optical_run.is_terminal:
        optical_run = await instrument.cancel_run(optical_run_id)
    if optical_run is None or optical_run.plate_release_state != "uncertain":
      await instrument.set_lamp(False)
      await instrument.set_filter_position(0)
    if optical_run is not None and optical_run.state in ("completed", "cancelled", "aborted"):
      plate = await instrument.release_plate(optical_run_id)
      print("Internal plate mechanism:", plate.state)
  finally:
    await instrument.stop()

## Read saved collection progress

This optional cell reads the journal without USB traffic or loading camera images. Select an
existing optical run ID. Records retain complete and uncertain collection intents; use
`get_collections()` when you need the original or partial frames. A complete checkpoint alone
does not establish thermal completion or validate image contents.

During collection, use this method on the same driver instance; it may wait until the current
burst releases the run lock. It also works without `setup()` when reading saved evidence.

In [ ]:
read_saved_optical_progress = False
saved_optical_run_id = "plr-optical-001"
if read_saved_optical_progress:
  records = await instrument.get_collection_records(saved_optical_run_id)
  print("Completed:", sum(record.state == "complete" for record in records))
  print([(record.ordinal, record.point, record.state, record.error) for record in records])

## Release the plate after the approved thermal-only session

This optional cleanup is for the thermal-only `run_id` above. Use it after normal completion or
explicit cancellation, with hands clear of the mechanism. The optical-session cell already
performs its own cleanup and disconnects. Enable only the section belonging to the approved run.

`release_plate()` requires a completed, cancelled or aborted checkpoint and checks fresh history,
inactivity and optical state. An uncertain release is only observed, never repeated; skip the
lamp/filter setters in that case. After a communication fault, first reconnect with the same
journal. Inspect failed loading or ambiguous ownership instead of changing IDs to force release.
See {ref}`plate release and recovery <abi7500-plate-release>`. The cover can remain hot.

The public release path passed a separate 30 °C/60-second lamp-off session. Repeat calls on the
same connection and after clean reconnect sent no additional release command. See the
{ref}`public plate-release report <abi7500-trayapi-validation>`. Ambiguous-failure recovery remains
hardware unverified.


In [ ]:
release_finished_thermal_plate = False
if release_finished_thermal_plate:
  try:
    run = await instrument.get_run(run_id)
    if run.state not in ("completed", "cancelled", "aborted"):
      raise RuntimeError("Resolve the run before releasing the plate")
    if run.plate_release_state != "uncertain":
      await instrument.set_lamp(False)
      await instrument.set_filter_position(0)
    plate = await instrument.release_plate(run_id, timeout=30)
    print("Internal plate mechanism:", plate.state)
  finally:
    await instrument.stop()

## Close the connection

`stop()` closes USB, ends keepalives and releases journal ownership. It does not cancel a run,
disable the lamp, park the filter, release the plate or cool the cover. Use the explicit cleanup
above for the approved session. Calling `stop()` again after that cleanup is harmless.
After a communication failure, close before reconnecting with the same journal. Reconcile the
same run or release attempt; do not replay Start or switch journals.

In [ ]:
await instrument.stop()

## Offline packet tools

The remaining cells only use saved bytes and fixtures. They never open an instrument connection.

In [ ]:
from pylabrobot.thermo_fisher.abi7500_fast_dx.protocol import (
  ThermalPacket,
  ThermalPacketDecoder,
  decode_serial_read,
  encode_serial_read,
  encode_serial_write,
)

## Decode a reference packet

This temperature-query packet was generated by the reference binary in an isolated CPU emulator. It is not a captured instrument response. The sequence value identifies a message, not an experiment run.

In [ ]:
reference = bytes.fromhex("5858ffffffff000f393730300006000000000001e6")
packet = ThermalPacket.decode(reference)
assert packet.command == 0x06
assert packet.sequence == 1
assert packet.payload == b""
assert packet.encode() == reference
packet

## Reassemble fragments

A thermal packet can span multiple tunnel replies. A strict decoder retains incomplete bytes and rejects malformed packets without guessing where the next one begins. `finish()` detects a truncated capture.

In [ ]:
decoder = ThermalPacketDecoder()
assert decoder.feed(reference[:9]) == []
reassembled = decoder.feed(reference[9:])
decoder.finish()
assert reassembled == [packet]
reassembled

## Inspect the USB serial wrapper

The recovered serial-write wrapper uses a four-character space-padded length and writes of at most 64 bytes. The serial-read request includes two bytes for the reply count. Inspect these bytes offline; the startup capture confirms the serial wrapper, with additional direct Ubuntu hardware evidence described in the hardware report.

In [ ]:
write_chunks = encode_serial_write(reference)
read_request = encode_serial_read(len(reference))
assert write_chunks == (b"RS  21" + reference,)
assert read_request == b"RR  23"
{"write_hex": [chunk.hex() for chunk in write_chunks], "read_request": read_request}

## Separate valid bytes from the USB reply tail

Captured RR replies fill the requested length even when fewer serial bytes are available. The little-endian count identifies the valid bytes; the unused tail can be nonzero. This synthetic example wraps the earlier reference packet with four unused bytes. It illustrates parsing only, not a real instrument response. Unmodified captured examples are covered by the protocol tests.

In [ ]:
wire_reply = len(reference).to_bytes(2, "little") + reference + b"\xff" * 4
serial_bytes = decode_serial_read(wire_reply, requested_size=len(reference) + 4)
assert serial_bytes == reference
{"wire_bytes": len(wire_reply), "valid_serial_bytes": len(serial_bytes)}

## Reconstruct a captured camera frame

The repository contains one compressed CD payload as a regression fixture. Its pixels are little-endian unsigned 16-bit values. `CameraFrameAssembler` preserves every byte and reproduces the native library's crop, including its one-pixel offset. This example uses the original USB read boundaries; it does not identify physical wells or apply calibration.

In [ ]:
import base64
import json
import zlib
from importlib.resources import files

from pylabrobot.thermo_fisher.abi7500_fast_dx import CameraFrameAssembler, CameraGeometry

fixture = json.loads(
  files("pylabrobot.thermo_fisher.abi7500_fast_dx")
  .joinpath("test_data/camera_capture_vector.json")
  .read_text()
)
wire_data = zlib.decompress(base64.b64decode(fixture["payload"]))
geometry = CameraGeometry.decode(bytes.fromhex(fixture["ci_reply_hex"]))
assembler = CameraFrameAssembler(geometry)
offset = 0
for length in fixture["usb_read_lengths"]:
  assembler.feed(wire_data[offset : offset + length])
  offset += length
frame = assembler.finish()
assert frame.wire_data == wire_data
{"width": frame.width, "height": frame.height, "first_pixels": frame.pixel_rows()[0][:8]}

## Read saved vendor filter values

EDS members ending in `.tiff` can contain 96 signed 64-bit well values rather than camera TIFFs. This fixture comes from the completed run; its expected values were checked against the vendor's separate live result log. Indices and integer precision are preserved. These are pre-uniformity vendor outputs, not uncorrected pixel sums. The camera processing API below reconstructs their reference acquisition configuration.

For a complete local EDS file, import `read_eds_filter_data` from the same package and call `read_eds_filter_data("/path/to/experiment.eds")`. It returns the available arrays in acquisition order; it does not open the instrument.

In [ ]:
from pylabrobot.thermo_fisher.abi7500_fast_dx import SavedFilterData

saved_fixture = json.loads(
  files("pylabrobot.thermo_fisher.abi7500_fast_dx")
  .joinpath("test_data/saved_filter_vector.json")
  .read_text()
)
saved = SavedFilterData.decode(saved_fixture["member"], bytes.fromhex(saved_fixture["data_hex"]))
assert list(saved.well_values) == saved_fixture["expected_values"]
{
  "stage": saved.stage,
  "cycle": saved.cycle,
  "filter": saved.filter,
  "first_values": saved.well_values[:8],
}

## Apply the experiment's saved uniformity calibration offline

Select a saved EDS file and its own embedded calibration. This preserves each original signed
integer and produces a separate uniformity-corrected filter value with binary32 precision.
The reference implementation matches the original vendor Java method bit-for-bit for 960 values.
A saved pass flag does not establish that a calibration remains current. Filter signals are not
dye concentrations or Ct values. Synchronized optical runs remain unimplemented. Diagnostic camera pixel statistics are a separate API.

In [ ]:
from pylabrobot.thermo_fisher.abi7500_fast_dx import (
  apply_uniformity,
  read_eds_filter_data,
  read_eds_uniformity,
)

experiment_path = Path("/path/to/experiment.eds")
calibration = read_eds_uniformity(experiment_path)
print(calibration.sha256, calibration.timestamp, calibration.instrument_id)
corrected = [
  apply_uniformity(source, calibration) for source in read_eds_filter_data(experiment_path)
]
if corrected:
  first = corrected[0]
  print(first.source.stage, first.source.cycle, first.source.filter)
  print(first.measurements[0])

## Diagnostic camera pixels by well

This optional offline cell reads an existing journaled acquisition and a mask from its matching calibration backup. Set the path and acquisition ID explicitly. The mask filter association is supplied by you; physical A1 orientation and the active instrument/calibration association remain unverified. The image is rotated 180° before applying the mask. Optical-black correction is a separate camera offset correction; the recorded vendor acquisition instead used OPB=0. These sums cannot be passed to `apply_uniformity`. See [measurement limits](optics-and-measurements.md).

In [ ]:
from pathlib import Path

from pylabrobot.thermo_fisher.abi7500_fast_dx import CameraWellData, PlatePixelMask

# Select existing local evidence; this cell performs no USB I/O.
mask_path = None  # Path("/path/to/ROIMask_W96_F0_I809_Commit.aoi")
if mask_path is not None:
  saved = await instrument.get_acquisition("plr-optics-001-f1")
  mask = PlatePixelMask.from_file(mask_path, filter=1)
  diagnostic = CameraWellData(saved, mask, correction="optical_black")
  for value in diagnostic.measurements:
    print(value.label, value.pixel_count, value.pixel_sum, value.mean)

## Reconstruct recorded camera frames into filter measurements

This optional cell performs no hardware I/O. Select a folder containing the matching original camera frames, F0 mask, native `Background.csv`, and completed EDS. The example uses reference cycle 1, physical filter 1 (saved filter 5). Replace filenames/exposures/metadata together when using another acquisition. The original files and intermediate pixel/ring statistics are retained. Check saturation flags before treating values as quantitative.

The native CSV has more coefficient precision than the EDS background INI. The supported configuration is OPB=0, 12-bit sum extraction with a one-pixel surrounding ring. It does not renew calibration or implement synchronized PCR acquisition. See [the processing guide](optics-and-measurements.md).

In [ ]:
from pylabrobot.thermo_fisher.abi7500_fast_dx import (
  BackgroundCalibration,
  CameraExposure,
  CameraFilterResult,
  CameraFrame,
  CameraGeometry,
  FluorescenceFrame,
  PlatePixelMask,
  read_eds_uniformity,
)


def recorded_frame(folder: Path, name: str, exposure: float, mask: PlatePixelMask) -> FluorescenceFrame:
  """Associate a recorded transfer with its original exposure and mask."""
  frame = CameraFrame(CameraGeometry(321, 245), (folder / name).read_bytes())
  return FluorescenceFrame(frame, CameraExposure(exposure), mask)


reference_folder = None  # Path("/path/to/reference-acquisition")
if reference_folder is not None:
  mask = PlatePixelMask.from_file(reference_folder / "ROIMask_W96_F0_I809_Commit.aoi", filter=1)
  result = CameraFilterResult(
    frames=(
      recorded_frame(reference_folder, "camera-016.bin", 0.540, mask),
      recorded_frame(reference_folder, "camera-017.bin", 0.180, mask),
      recorded_frame(reference_folder, "camera-018.bin", 0.060, mask),
    ),
    background=BackgroundCalibration.from_file(reference_folder / "Background.csv"),
    uniformity=read_eds_uniformity(reference_folder / "acquire-complete.eds"),
    stage=2, cycle=1, collection_point=1,
  )
  for value in result.measurements:
    print(result.filter, value.label, value.raw_value, value.corrected_value,
          value.selection.selected_index, value.selected_saturated)


## Export a completed optical run

This optional cell reads the same local journal without opening USB. Set paths only after the
session is finished and cleaned up. Supply the calibration files used for that acquisition;
this does not certify their current validity or physical well orientation. The ZIP contains
original camera transfers, calibration bytes, run/collection metadata, a selected-measurement
CSV, all exposure candidates in another CSV, and content hashes. It rejects incomplete runs
and existing destinations. Keep full archives outside Git. See the
{ref}`export guide <abi7500-export>` for field meanings.


In [ ]:
from pathlib import Path

from pylabrobot.thermo_fisher.abi7500_fast_dx import (
  BackgroundCalibration, PlatePixelMask, UniformityCalibration,
)

# No USB operations. Set these paths to explicitly associated saved evidence.
export_destination = None  # Path.home() / "Downloads" / "plr-cychome-001-results.zip"
calibration_folder = None  # Path("/path/to/instrument/calibration")
mask_folder = None  # Path("/path/to/matching/masks")
if export_destination is not None and calibration_folder is not None and mask_folder is not None:
  saved_archive = await instrument.export_run(
    "plr-cychome-001",
    export_destination,
    background=BackgroundCalibration.from_file(calibration_folder / "Background.csv"),
    uniformity=UniformityCalibration.from_file(calibration_folder / "uniformity.ini"),
    masks=tuple(
      PlatePixelMask.from_file(mask_folder / f"ROIMask_W96_F{p-1}_I809_Commit.aoi", filter=p)
      for p in range(1, 6)
    ),
  )
  print(saved_archive)


## Remaining hardware work

Ubuntu PLR has verified connection/status, thermal start/cancel, a completed 35 °C/60-second hold,
illuminated snapshots, and two normally completed 35 °C/30 °C cycles with 30 synchronized images.
The completed optical run used a 900-second observation budget and an already-warm cover.
Broader thermal programs, cold-cover startup with this budget, fault recovery, broader physical-label/channel coverage
and matching vendor comparisons of unsaturated measurements remain to be checked. Current calibration validity is not established.
Dye unmixing, Ct and assay interpretation remain outside this implementation.

A shorter-exposure session used 30/10/5 ms at physical positions 2–5 and produced no selected saturation flags
among 960 results. However, fixed-exposure signals at physical positions 4 and 5 declined by roughly 10% and
8% between cycles. Quantitative stability and the cause of that decline remain unresolved.
A constant-30 °C comparison also completed normally with 30 images and no selected saturation
flags. Median fixed-exposure signals still declined about 6.1% in vendor channel 1 (SYBR/FAM,
physical position 5) and 7.5% in vendor channel 2 (physical position 4). Thus the intervening
35 °C excursions are not required for the decline; dye/plate and illumination effects remain
unresolved. Channel assignments do not constitute spectral unmixing.
See the [hardware report](hardware-validation.md).

An approved constant-30 °C session with `park_between_collections=True` retained lamp power 1 and verified stationary position 0 after each burst. All 30 scheduled frames and 960 unflagged selections were saved; matched SYBR-channel decline was 0.8%, versus 6.1% in the earlier unparked session. Eight diagnostic frames supported camera-path suppression at position 0, not a measurement of sample irradiance. This supports an optical-state contribution without proving photobleaching. The vendor’s `0x67` home operation differs from this absolute move and its alignment effect remains untested. See the {ref}`parking validation <abi7500-parked-validation>`.

A subsequent approved four-collection, 60-image session verified native homing after collections 2 and 4, stationary position 0 and unchanged lamp power, with normal thermal completion. Homes took about 6 seconds versus 3.5 seconds for parking. All 1,920 selected measurements were unflagged. SYBR-channel declines were 0.82% after park, 0.47% after home and 0.43% after park again; this fixed-order comparison does not establish an added alignment or stability benefit from homing. Homing remains an explicit option. See {ref}`homing validation <abi7500-homing-validation>`.

A combined two-cycle 35 °C/30 °C session with homing after both collections and the updated
progress scheduler completed normally: 30 images, 960 unflagged selected measurements and
verified lamp-off/filter-0 cleanup. The SYBR-channel decline was 0.95%; differing lamp warmup
and plate history limit comparison with earlier sessions. See the
{ref}`combined cycling/homing validation <abi7500-cycling-home-validation>`.


A later approved ten-cycle 35 °C/30 °C session collected 30 images in the SYBR channel only,
verified homing after every three-image burst, and exercised the portable export with no USB
traffic. See the {ref}`ten-cycle report <abi7500-sybr10-validation>` for timing and measured
signal trends. It does not validate hotter programs, current calibration or physical well orientation.

A known-pattern fluorescein plate verified the physical orientation and labels **A1, A4, C9, F3, H7 and H12**
in physical filter 5. All six were the brightest wells at 5, 10 and 30 ms; the two shorter images
were unsaturated. Four marker wells saturated at 30 ms, and the selected results used shorter
exposures where needed. This verifies six physical labels on this plate, not every well/channel
or current calibration. The session completed normally and released the plate before closing USB;
its cover remained hot. See the {ref}`known-pattern report <abi7500-wellmap-validation>`.


## Raw measurements from explicitly supplied regions — offline

`RawCameraWellData` measures a saved acquisition with a diagnostic region map for the same
camera and physical filter. Regions remain provisional; values are uncalibrated camera counts.
A verified physical label does not validate its region boundary or all other labels. This cell
is disabled until both saved paths are supplied; it performs no instrument operations.
See {ref}`measurement processing <abi7500-raw-regions>`
and the [PCR notebook](pcr-run.ipynb) for complete-run CSV export and fixed-exposure plots.

In [ ]:
import json
from pathlib import Path
from pylabrobot.thermo_fisher.abi7500_fast_dx import (
  CameraAcquisition, DiagnosticPlateRegions, RawCameraWellData,
)

acquisition_path = None  # Saved acquisition JSON containing the original wire_data.
regions_path = None  # Explicit diagnostic regions for the same camera and physical filter.
if acquisition_path is not None and regions_path is not None:
  acquisition = CameraAcquisition.from_dict(json.loads(Path(acquisition_path).read_text()))
  regions = DiagnosticPlateRegions.from_dict(json.loads(Path(regions_path).read_text()))
  for value in RawCameraWellData(acquisition, regions).measurements:
    print(value.label, value.mean, value.clipped_pixel_count, value.label_verified)